# Simple filter

In [1]:
import math

import rclpy
from rclpy.node import Node

from geometry_msgs.msg import PointStamped, PoseStamped
from transformations import quaternion_from_euler


In [2]:
class PointToPoseNode(Node):
    def __init__(self):
        super().__init__('point_to_pose_node')

        self.sub = self.create_subscription(
            PointStamped,
            'point_stamped',
            self.point_callback,
            10
        )

        self.pub = self.create_publisher(
            PoseStamped,
            'pose',
            10
        )

        self.prev_point = None
        self.prev_yaw = None

        # Low-pass filter parameter
        # alpha in (0, 1): smaller = more smoothing
        self.alpha = 0.05

    def point_callback(self, msg: PointStamped):
        # First message: store and return
        if self.prev_point is None:
            self.prev_point = msg
            return

        dx = msg.point.x - self.prev_point.point.x
        dy = msg.point.y - self.prev_point.point.y

        # Ignore very small motion to avoid noise
        if abs(dx) < 1e-6 and abs(dy) < 1e-6:
            return

        raw_yaw = math.atan2(dy, dx)

        # Initialize filter
        if self.prev_yaw is None:
            filtered_yaw = raw_yaw
        else:
            # Handle angle wrapping correctly
            yaw_error = math.atan2(
                math.sin(raw_yaw - self.prev_yaw),
                math.cos(raw_yaw - self.prev_yaw)
            )
            filtered_yaw = self.prev_yaw + self.alpha * yaw_error

        qx, qy, qz, qw = quaternion_from_euler(filtered_yaw, 0.0, 0.0)

        pose = PoseStamped()
        pose.header = msg.header

        # Position from current point
        pose.pose.position.x = msg.point.x
        pose.pose.position.y = msg.point.y
        pose.pose.position.z = msg.point.z

        # Orientation from motion direction
        pose.pose.orientation.x = qx
        pose.pose.orientation.y = qy
        pose.pose.orientation.z = qz
        pose.pose.orientation.w = qw

        self.pub.publish(pose)

        # Update previous point
        self.prev_point = msg

In [ ]:
rclpy.init()
node = PointToPoseNode()
rclpy.spin(node)
node.destroy_node()
rclpy.shutdown()

# Window sliding Filter

In [1]:
import math
from collections import deque

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import PointStamped, PoseStamped
from transformations import quaternion_from_euler

In [2]:
class PointToPoseSlidingWindow(Node):
    def __init__(self):
        super().__init__('point_to_pose_sliding_window')

        self.sub = self.create_subscription(
            PointStamped,
            'human_position_map',
            self.callback,
            10
        )

        self.pub = self.create_publisher(
            PoseStamped,
            'human_pose',
            10
        )

        # Sliding window of recent points
        self.window_size = 5
        self.points = deque(maxlen=self.window_size)

        # Minimum displacement to consider valid motion
        self.min_motion = 1e-4

        # Motion detection
        # self.motion_threshold = 7e-2  # meters
        self.motion_threshold = 9e-2  # meters

        # Default yaw when still (e.g. facing +X)
        self.default_yaw = 0.0  # radians

    # ---------- Motion check ----------
    def is_moving(self) -> bool:
        """
        Returns True if displacement over the window
        exceeds the motion threshold.
        """
        if len(self.points) < 2:
            return False

        p0 = self.points[0].point
        pN = self.points[-1].point

        dx = pN.x - p0.x
        dy = pN.y - p0.y
        dz = pN.z - p0.z

        displacement = math.sqrt(dx*dx + dy*dy)
        return displacement > self.motion_threshold
        # self.get_logger().info(f"dx: {dx} dy: {dy}")
        # return abs(dx) > self.motion_threshold and abs(dy) > self.motion_threshold
        
    def callback(self, msg: PointStamped):
        self.points.append(msg)

        # Need at least 2 points
        if len(self.points) < 2:
            return

        # # Use first and last point in window
        # p0 = self.points[0].point
        # pN = self.points[-1].point

        # dx = pN.x - p0.x
        # dy = pN.y - p0.y

        # if abs(dx) < self.min_motion and abs(dy) < self.min_motion:
        #     return

        # yaw = math.atan2(dy, dx)

        latest = self.points[-1].point
        
        # Estimate yaw from sliding window
        p0 = self.points[0].point
        pN = latest
        
        if self.is_moving():
            dx = pN.x - p0.x
            dy = pN.y - p0.y
            yaw = math.atan2(dy, dx)
        else:
            # Force stable default orientation
            yaw = self.default_yaw
            
        qx, qy, qz, qw = quaternion_from_euler(yaw, 0.0, 0.0)

        pose = PoseStamped()
        pose.header = msg.header

        # Position = latest point
        pose.pose.position.x = pN.x
        pose.pose.position.y = pN.y
        pose.pose.position.z = pN.z

        pose.pose.orientation.x = qx
        pose.pose.orientation.y = qy
        pose.pose.orientation.z = qz
        pose.pose.orientation.w = qw

        self.pub.publish(pose)

In [ ]:
rclpy.init()
node = PointToPoseSlidingWindow()
rclpy.spin(node)


In [ ]:
node.destroy_node()
rclpy.shutdown()

In [ ]:
node.points